# 🌳 Random Forest — Complete Concepts Guide
### (Synthetic Sample Data for Clear Understanding)

---

## 👋 Welcome!

This notebook is built **step by step** for complete beginners.
We use a **simple, made-up dataset** so you can focus on *understanding the concept*, not on cleaning messy real data.

---

## 📚 What You Will Learn

| # | Topic |
|---|-------|
| 1 | What is Random Forest? (with real-life analogy) |
| 2 | Ensemble Learning — combining many models |
| 3 | Bootstrapping & Bagging |
| 4 | Out-of-Bag (OOB) Error — free validation! |
| 5 | Key Hyperparameters — `n_estimators`, `max_depth`, `max_features`, etc. |
| 6 | Effect of `n_estimators` and `max_features` on accuracy |
| 7 | Random Forest **Classifier** — Step by step |
| 8 | Confusion Matrix — **Old Way** (seaborn) + **New Way** (ConfusionMatrixDisplay) |
| 9 | Classification Report, ROC-AUC Curve |
| 10 | Feature Importance |
| 11 | Hyperparameter Tuning with **GridSearchCV** |
| 12 | Random Forest **Regressor** |
| 13 | Regression Metrics — MAE, MSE, RMSE, R² |

---

> 💡 **Every new technical word is explained before it is used!**
> 🎯 **Goal:** By the end, you should be able to build your own Random Forest model from scratch!

In [ ]:
# ═══════════════════════════════════════════════════
# 📦  IMPORTING LIBRARIES
# ═══════════════════════════════════════════════════
# numpy    ─ numerical calculations & random data
# pandas   ─ working with tables (DataFrames)
# matplotlib / seaborn ─ drawing charts
# sklearn  ─ all Machine Learning tools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Random Forest models ──────────────────────────
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# ── Splitting & Tuning ────────────────────────────
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score

# ── Preprocessing ─────────────────────────────────
from sklearn.preprocessing import StandardScaler, LabelEncoder

# ── Classification Metrics ────────────────────────
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,   # NEW built-in way (sklearn ≥ 0.24)
    classification_report,
    roc_auc_score, roc_curve, auc
)

# ── Regression Metrics ────────────────────────────
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

print("✅ All libraries imported successfully!")

## 🌳 Chapter 1: What is Random Forest?

---

### 📖 Simple Definition
> **Random Forest** is a Machine Learning algorithm that creates many **Decision Trees** and combines their predictions to give one final, reliable answer.

---

### 🎓 Real-Life Analogy — "The Expert Panel"

Imagine you are sick and you want to know what disease you have.

| Option | Approach | Problem |
|--------|----------|---------|
| **Bad** | Ask ONE doctor | That one doctor might be wrong |
| **Smart** | Ask **100 different doctors**, each looks at *different symptoms* | Take the **majority vote** → More reliable! |

👉 **This is exactly what Random Forest does!**

| Random Forest | Doctor Analogy |
|--------------|----------------|
| Each **Decision Tree** | = One Doctor |
| Each tree looks at **random subset of data** | = Each doctor checks different symptoms |
| **Final prediction** | = Majority Vote (classification) or Average (regression) |

---

### 🏗️ How Random Forest Works — Step by Step

```
Original Data
     │
     ├─── Bootstrap Sample 1 ──► Tree 1 ──► Prediction 1
     ├─── Bootstrap Sample 2 ──► Tree 2 ──► Prediction 2
     ├─── Bootstrap Sample 3 ──► Tree 3 ──► Prediction 3
     │         ...                  ...          ...
     └─── Bootstrap Sample N ──► Tree N ──► Prediction N
                                               │
                              Classification: Majority Vote
                              Regression:     Average
```

---

### 💡 Why Random Forest > Single Decision Tree?

| Problem with ONE Tree | How Random Forest Solves It |
|-----------------------|-----------------------------|
| **Overfits** (memorizes training data) | Many trees on random samples reduce overfitting |
| **High variance** (unstable) | Averaging reduces variance |
| **One bad split = bad model** | Many trees correct each other's mistakes |

## 🤝 Chapter 2: Ensemble Learning

---

### 📖 Definition
> **Ensemble Learning** = Combining multiple machine learning models to get better predictions than any single model alone.

---

### 🌟 Types of Ensemble Methods

| Method | Strategy | Example |
|--------|----------|---------|
| **Bagging** | Train models on different random samples → combine | **Random Forest** ✅ |
| **Boosting** | Train sequentially; each model fixes the previous one's errors | XGBoost, AdaBoost |
| **Stacking** | Use predictions of multiple models as input to another | StackingClassifier |

---

### 🔑 Key Concept — Bias vs Variance

| Term | Meaning | Causes |
|------|---------|--------|
| **Bias** | Model is too simple, misses patterns | Underfitting |
| **Variance** | Model is too sensitive to training data | Overfitting |

> 🎯 **Random Forest** mainly reduces **Variance** by averaging many trees.
> This is why it generalises better than a single Decision Tree!

## 📂 Creating Our Sample Dataset

---

### 🎓 Scenario: Student Exam Performance Prediction

A school wants to predict if a student will **PASS** or **FAIL** based on their habits and history.

| Column | Description | Range |
|--------|-------------|-------|
| `study_hours` | Hours studied per day | 1 – 10 |
| `sleep_hours` | Hours of sleep per night | 5 – 9 |
| `attendance_pct` | % of classes attended | 60 – 100 |
| `previous_score` | Score in last exam | 40 – 100 |
| `extracurricular` | Does extra activities? | 0 = No, 1 = Yes |
| `passed` *(target)* | Did student pass? | 0 = Fail, 1 = Pass |

In [ ]:
# ═══════════════════════════════════════════════════
# 📊  CREATE STUDENT PERFORMANCE DATASET
# ═══════════════════════════════════════════════════

np.random.seed(42)       # Fixed seed = same results every time you run
n = 300                  # 300 students

study_hours    = np.random.uniform(1, 10, n)
sleep_hours    = np.random.uniform(5,  9,  n)
attendance_pct = np.random.uniform(60, 100, n)
previous_score = np.random.uniform(40, 100, n)
extracurricular= np.random.randint(0, 2,   n)

df = pd.DataFrame({
    'study_hours'    : np.round(study_hours, 1),
    'sleep_hours'    : np.round(sleep_hours, 1),
    'attendance_pct' : np.round(attendance_pct, 1),
    'previous_score' : np.round(previous_score, 1),
    'extracurricular': extracurricular
})

# Target: student passes if weighted score > 0.5
score = (0.4 * study_hours / 10 +
         0.35 * attendance_pct / 100 +
         0.25 * previous_score / 100)
df['passed'] = (score + np.random.normal(0, 0.08, n) > 0.5).astype(int)

print("=" * 55)
print("✅ Dataset Created!")
print("=" * 55)
print(f"Total Students : {len(df)}")
print(f"Passed (1)     : {df['passed'].sum()}  ({df['passed'].mean()*100:.1f}%)")
print(f"Failed (0)     : {(df['passed']==0).sum()}  ({(1-df['passed'].mean())*100:.1f}%)")
print(f"\nColumns: {list(df.columns)}")

## 📂 Step 1: Data Loading — Looking at Our Data

---

### 🔑 New Words

| Term | Meaning |
|------|---------|
| **DataFrame** | A table with rows and columns (like Excel) |
| **Shape** | `(rows, columns)` — dimensions of the table |
| `head()` | Shows first 5 rows |
| `dtypes` | Data type of each column (`int64`, `float64`, `object`) |

---

> 🎯 **What we learn here**: Always look at your data FIRST before doing anything!

In [ ]:
# ═══════════════════════════════════════════════════
# 👀  STEP 1 — LOOK AT THE DATA
# ═══════════════════════════════════════════════════

print("🔍 First 5 rows:")
display(df.head())

print(f"\n📐 Shape: {df.shape}")
print(f"   → {df.shape[0]} rows (students) × {df.shape[1]} columns (features)")

print("\n📋 Column Data Types:")
print(df.dtypes)

## 🔍 Step 2: Exploratory Data Analysis (EDA)

---

### 📖 What is EDA?
> **EDA** = Looking at and understanding your data *before* building a model.

### 🎯 Why Do We Do EDA?
1. Find **missing values** (empty/null cells)
2. Understand **distributions** (how values are spread)
3. Find **relationships** between features (correlations)
4. Spot **outliers** (unusual data points)

---

### 🔑 New Words

| Term | Meaning |
|------|---------|
| **Missing Values** | Empty / null cells — the model can't handle these! |
| **Distribution** | How values are spread (many small, few large, etc.) |
| **Correlation** | How much two columns are related (range: −1 to +1) |
| **Outlier** | A data point that is very far from the others |

---

> 🎯 **What we learn here**: Understand our data shape, missing values, and feature relationships.

In [ ]:
# ═══════════════════════════════════════════════════
# 📈  STATISTICAL SUMMARY
# ═══════════════════════════════════════════════════
# describe() shows: count, mean, std, min, 25%, 50%, 75%, max

print("📊 Statistical Summary of All Columns:")
display(df.describe())

print("\n📋 Dataset Info (data types + non-null counts):")
df.info()

In [ ]:
# ─── Missing Values Check ────────────────────────────────────
print("❓ Missing Values in Each Column:")
print("-" * 35)
missing = df.isnull().sum()
print(missing)
print(f"\n{'✅ No missing values!' if missing.sum()==0 else f'⚠️ Total missing: {missing.sum()}'}")

In [ ]:
# ─── Target Variable Distribution ───────────────────────────
print("🎯 Target Variable (passed) — Value Counts:")
print(df['passed'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
df['passed'].value_counts().plot(kind='bar', ax=axes[0],
    color=['salmon','lightgreen'], edgecolor='black', width=0.5)
axes[0].set_title('Pass vs Fail — Count', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(['Failed (0)', 'Passed (1)'], rotation=0)
axes[0].set_ylabel('Number of Students')

# Pie chart
df['passed'].value_counts().plot(kind='pie', ax=axes[1],
    labels=['Passed','Failed'], autopct='%1.1f%%',
    colors=['lightgreen','salmon'], startangle=90)
axes[1].set_title('Pass vs Fail — %', fontsize=12, fontweight='bold')
axes[1].set_ylabel('')

plt.suptitle('Target Variable Distribution', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ─── Correlation Heatmap ─────────────────────────────────────
# Correlation: +1 = perfect positive, 0 = no relation, -1 = perfect negative
plt.figure(figsize=(8, 5))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Heatmap\n(How related are the features?)',
          fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

corr_with_target = df.corr()['passed'].drop('passed').abs().sort_values(ascending=False)
print("\n📌 Features ranked by correlation with 'passed':")
print(corr_with_target.to_string())

In [ ]:
# ─── Feature Distributions: Passed vs Failed ─────────────────
features = ['study_hours', 'sleep_hours', 'attendance_pct', 'previous_score']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for i, feat in enumerate(features):
    ax = axes[i//2, i%2]
    df[df['passed']==1][feat].hist(ax=ax, bins=20, alpha=0.6,
                                    label='Passed', color='green')
    df[df['passed']==0][feat].hist(ax=ax, bins=20, alpha=0.6,
                                    label='Failed', color='red')
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel(feat); ax.set_ylabel('Count')
    ax.legend()

plt.suptitle('Feature Distributions: Passed vs Failed',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("💡 If green and red bars are SEPARATED → that feature is a GOOD predictor!")

## 🔧 Step 3: Data Preprocessing

---

### 📖 What is Preprocessing?
> **Preprocessing** = Preparing and cleaning data so the model can learn from it correctly.

---

### 🔑 New Words

| Term | Meaning |
|------|---------|
| **Features (X)** | Input columns (what we give the model to learn from) |
| **Target (y)** | Output column (what we want the model to predict) |
| **Train Set** | 80% of data — used to *teach* the model |
| **Test Set** | 20% of data — used to *evaluate* the model on unseen data |
| `random_state` | A fixed number so your split is the same every time |
| `stratify=y` | Ensures equal pass/fail ratio in both train and test |

---

> 🎯 **What we learn here**: Split data properly so we can evaluate the model fairly.

In [ ]:
# ═══════════════════════════════════════════════════
# ✂️  STEP 3 — TRAIN / TEST SPLIT
# ═══════════════════════════════════════════════════

# Separate features and target
X = df.drop('passed', axis=1)   # Everything except 'passed'
y = df['passed']                  # Only 'passed'

print(f"Features (X): {X.shape}  → columns: {list(X.columns)}")
print(f"Target   (y): {y.shape}")

# Split  —  80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size   = 0.2,
    random_state= 42,
    stratify    = y      # keeps pass/fail ratio equal in both splits
)

print(f"\n✅ Split complete!")
print(f"   Training: {X_train.shape[0]} students ({X_train.shape[0]/len(df)*100:.0f}%)")
print(f"   Testing : {X_test.shape[0]}  students ({X_test.shape[0]/len(df)*100:.0f}%)")
print(f"\n   Train pass rate: {y_train.mean()*100:.1f}%")
print(f"   Test  pass rate: {y_test.mean()*100:.1f}%  ← should be similar (stratify worked!)")

## 🌲 Prerequisite: Decision Trees — The Building Blocks

---

### 📖 What is a Decision Tree?
> A **Decision Tree** is a flowchart that makes decisions by asking Yes/No questions about the data.

---

### 🎯 Example — Will a Student Pass?

```
            Is study_hours > 5?
           /                   \
         YES                   NO
          |                     |
 Is attendance_pct > 75%?    ❌ Predict FAIL
       /         \
     YES          NO
      |            |
  ✅ PASS       ❌ FAIL
```

---

### ⚠️ Problem with a Single Decision Tree

1. **Overfitting** — Memorises training data, fails on new data
2. **High Variance** — Small change in data = completely different tree
3. **Unstable** — Not reliable for important decisions

---

### 💡 Solution = Random Forest!

Build **many trees** (each slightly different) and combine their answers!

---

### 🔑 New Words

| Term | Meaning |
|------|---------|
| **Gini Impurity** | How "mixed" a node is. 0 = pure (one class). 0.5 = most mixed. Formula: `Gini = 1 − Σp²` |
| **Node** | A decision point in the tree |
| **Leaf** | The final answer at the end of a branch |
| **Depth** | How many levels deep a tree goes |

## 🥾 Chapter 3: Bootstrapping

---

### 📖 Definition
> **Bootstrapping** = Creating new datasets by randomly sampling FROM the original dataset **WITH REPLACEMENT**.

---

### 🤔 "With Replacement" — What Does It Mean?

Imagine you have 10 cards (numbered 1–10) in a box.
1. Pick a card, **write the number**, then **put the card BACK**
2. Pick again — you might get the same card!
3. Repeat N times

This is **"with replacement"** — the same item can be picked more than once.

---

### 📊 Bootstrapping in Numbers

If original data has **N = 300 students**:
- Bootstrap sample also has **N = 300 rows**
- But only about **~63%** are unique (some students appear 2–3 times)
- The other **~37%** that were never picked = **Out-of-Bag (OOB) samples**

---

### 🎯 Why Do We Bootstrap?
- Each tree needs **slightly different training data** to become unique
- Unique trees → diverse predictions → better combined result
- Bootstrapping gives us many different datasets from ONE original dataset!

In [ ]:
# ═══════════════════════════════════════════════════
# 🥾  BOOTSTRAPPING DEMO
# ═══════════════════════════════════════════════════

print("Original Dataset: Students 1 to 10")
original = list(range(1, 11))
print(f"  {original}  (10 students)")

print("\n── Creating Bootstrap Samples (sampling WITH replacement) ──")

np.random.seed(42)
for i in range(1, 4):
    sample = sorted(np.random.choice(original, size=10, replace=True))
    unique = sorted(set(sample))
    oob    = [x for x in original if x not in sample]
    print(f"\nBootstrap Sample {i}: {sample}")
    print(f"  Unique students : {unique} ({len(unique)} unique = {len(unique)*10}%)")
    print(f"  OOB students    : {oob}   ← NOT used to train Tree {i}")

print("\n" + "="*55)
print("KEY INSIGHT:")
print("  Each tree trains on DIFFERENT students")
print("  → Diversity among trees → Better Random Forest!")
print("="*55)

## 📊 Chapter 4: Out-of-Bag (OOB) Error

---

### 📖 Definition
> **OOB (Out-of-Bag) Error** = The error calculated using samples that were **NOT** used to train a particular tree.

---

### 🎯 Why is OOB Useful?

Normally you need a **separate validation set** to check if your model is good.
With OOB, you get a **free validation score** using the ~37% samples each tree never saw!

---

### 🔄 How OOB Score is Calculated

```
For Student 42:
  Trees that did NOT use Student 42 = Tree 3, Tree 7, Tree 15 ...
  → These trees predict: Pass, Pass, Fail ...
  → Majority vote = Pass

Compare with actual result:
  Student 42 actually Passed → Correct! ✅

Repeat for all students → OOB Accuracy
```

---

### 🔑 Key Parameter
To enable OOB scoring: `RandomForestClassifier(oob_score=True)`
Access result: `rf_model.oob_score_`

In [ ]:
# ═══════════════════════════════════════════════════
# 📊  OOB SCORE DEMO
# ═══════════════════════════════════════════════════

rf_oob = RandomForestClassifier(
    n_estimators = 100,
    oob_score    = True,   # ← Enable OOB scoring
    random_state = 42
)
rf_oob.fit(X_train, y_train)

print("=" * 50)
print("OOB Score Demonstration")
print("=" * 50)
print(f"OOB Score (like free validation) : {rf_oob.oob_score_*100:.2f}%")
print(f"Actual Test Accuracy             : {rf_oob.score(X_test, y_test)*100:.2f}%")
print("=" * 50)
print("\n💡 If OOB ≈ Test Accuracy → Model is NOT overfitting ✅")
print("   If OOB << Test Accuracy → Model is OVERFITTING ❌")

## 🏗️ Step 4: Building a Random Forest Classifier

---

### 📖 What is Classification?
> **Classification** = Predicting which **category/class** an item belongs to.

**Our Example:** Predict whether a student will **Pass (1)** or **Fail (0)**.

---

### 🌳 The Three Steps

```python
# Step 1 — Create the model
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Step 2 — Train (teach the model)
rf.fit(X_train, y_train)

# Step 3 — Predict
y_pred = rf.predict(X_test)
```

---

> 🎯 **What we learn here**: How to create, train, and get predictions from a Random Forest model.

In [ ]:
# ═══════════════════════════════════════════════════
# 🌳  BUILD RANDOM FOREST CLASSIFIER
# ═══════════════════════════════════════════════════

# Step 1: Create model
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)

# Step 2: Train
rf_clf.fit(X_train, y_train)

# Step 3: Predict
y_pred = rf_clf.predict(X_test)

# Step 4: Accuracy
train_acc = rf_clf.score(X_train, y_train)
test_acc  = accuracy_score(y_test, y_pred)

print("=" * 50)
print("🌲 RANDOM FOREST CLASSIFIER — RESULTS")
print("=" * 50)
print(f"Training Accuracy : {train_acc*100:.2f}%")
print(f"Testing  Accuracy : {test_acc*100:.2f}%")
print("=" * 50)
print("\n📋 First 10 Predictions:")
print(f"  Actual   : {list(y_test.values[:10])}")
print(f"  Predicted: {list(y_pred[:10])}")

## ⚙️ Chapter 5: Key Hyperparameters

---

### 📖 What are Hyperparameters?
> **Hyperparameters** = Settings you choose **BEFORE** training. The model does NOT learn them from data.

---

### 🔑 Important Hyperparameters (Must Know!)

| Hyperparameter | Default | What it Controls |
|----------------|---------|-----------------|
| `n_estimators` | 100 | **Number of trees** — more = more accurate (but slower) |
| `max_depth` | `None` | **Max depth of each tree** — `None` = tree grows fully |
| `max_features` | `'sqrt'` | **Features at each split** — subset creates diversity |
| `min_samples_split` | 2 | **Min samples to split a node** — higher = simpler tree |
| `min_samples_leaf` | 1 | **Min samples at each leaf** — higher = smoother predictions |
| `bootstrap` | `True` | **Use bootstrapping?** — `False` = use all data for each tree |
| `oob_score` | `False` | **Enable OOB validation?** — `True` = free validation score |
| `random_state` | `None` | **Seed for randomness** — set for reproducibility |
| `n_jobs` | 1 | **Parallel CPU cores** — `-1` = use ALL available CPUs |
| `class_weight` | `None` | **Handle imbalanced classes** — `'balanced'` auto-adjusts |

---

### 💡 Tuning Priority
> Start with defaults → Then tune: `n_estimators` → `max_depth` → `max_features`

In [ ]:
# ═══════════════════════════════════════════════════
# ⚙️  HYPERPARAMETER EXPLORATION
# ═══════════════════════════════════════════════════

configs = [
    dict(n_estimators=10,  max_depth=None, max_features='sqrt'),
    dict(n_estimators=100, max_depth=None, max_features='sqrt'),
    dict(n_estimators=100, max_depth=3,    max_features='sqrt'),
    dict(n_estimators=100, max_depth=10,   max_features='sqrt'),
    dict(n_estimators=100, max_depth=None, max_features=2),
    dict(n_estimators=100, max_depth=None, max_features=None),
]

rows = []
for cfg in configs:
    m = RandomForestClassifier(**cfg, random_state=42)
    m.fit(X_train, y_train)
    rows.append({**cfg,
                 'Train %': round(m.score(X_train, y_train)*100, 1),
                 'Test %' : round(m.score(X_test,  y_test )*100, 1)})

result_df = pd.DataFrame(rows)
print("Hyperparameter Comparison:")
display(result_df)
print("\n💡 Notes:")
print("  max_depth=None → Full tree → can overfit (Train% >> Test%)")
print("  max_depth=3    → Very shallow → may underfit")
print("  max_features=None → Use all features → less diverse trees")

## 🌲 Deep Dive: `n_estimators` and `max_features`

---

### 🌳 `n_estimators` — Number of Trees

| Trees | Speed | Accuracy |
|-------|-------|----------|
| 10 | Very Fast | Low |
| 100 | Medium | Good ✅ |
| 500+ | Slow | Slightly better (diminishing returns) |

> **Rule of Thumb**: Start with 100–200. After that, accuracy improvements become very small.

---

### 🎲 `max_features` — Features Used at Each Split

This is what makes Random Forest truly **random**!

At each node split, instead of checking ALL features, only a **random subset** is considered.

| Value | Features Used | Use When |
|-------|---------------|----------|
| `'sqrt'` | √(total features) | **Classification** (default) ✅ |
| `'log2'` | log₂(total features) | Alternative for classification |
| `None` | ALL features | No randomness — like Bagging |
| `int (e.g. 2)` | Exactly 2 | Custom |
| `float (e.g. 0.5)` | 50% of features | Custom percentage |

---

### 💡 Why Limit Features?
- Using ALL features → Trees are similar → Less diversity → Worse ensemble
- Fewer features → Trees are **diverse** → **Better ensemble** (the Random Forest magic!)

> 🔑 `max_features` controls the **diversity vs individual accuracy trade-off**

In [ ]:
# ─── Effect of n_estimators ──────────────────────────────────
n_range = [5, 10, 25, 50, 100, 150, 200, 300]
tr_scores, te_scores = [], []

for n in n_range:
    rf = RandomForestClassifier(n_estimators=n, random_state=42)
    rf.fit(X_train, y_train)
    tr_scores.append(rf.score(X_train, y_train))
    te_scores.append(rf.score(X_test,  y_test))

plt.figure(figsize=(10, 5))
plt.plot(n_range, tr_scores, 'b-o', label='Train Accuracy', linewidth=2)
plt.plot(n_range, te_scores, 'r-o', label='Test  Accuracy', linewidth=2)
plt.xlabel('n_estimators (Number of Trees)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Effect of n_estimators on Accuracy', fontsize=13, fontweight='bold')
plt.legend(); plt.grid(True, alpha=0.3); plt.xticks(n_range)
plt.tight_layout(); plt.show()

print("💡 After ~100 trees, accuracy improvement becomes very small.")
print("   This is called 'Diminishing Returns'.")

In [ ]:
# ─── Effect of max_features ──────────────────────────────────
mf_vals   = [1, 2, 3, 4, 5, 'sqrt', 'log2', None]
mf_labels = ['1', '2', '3', '4', '5', 'sqrt', 'log2', 'None']
tr_mf, te_mf = [], []

for mf in mf_vals:
    rf = RandomForestClassifier(n_estimators=100, max_features=mf, random_state=42)
    rf.fit(X_train, y_train)
    tr_mf.append(rf.score(X_train, y_train))
    te_mf.append(rf.score(X_test,  y_test))

x = range(len(mf_vals))
plt.figure(figsize=(10, 5))
plt.plot(x, tr_mf, 'b-o', label='Train Accuracy', linewidth=2)
plt.plot(x, te_mf, 'r-o', label='Test  Accuracy', linewidth=2)
plt.xticks(x, mf_labels)
plt.xlabel('max_features Value', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title(f'Effect of max_features  (we have {X_train.shape[1]} total features)',
          fontsize=13, fontweight='bold')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"\n💡 Total features: {X_train.shape[1]}")
print(f"   'sqrt' uses ≈ {int(np.sqrt(X_train.shape[1]))} features per split")
print("   Fewer features = more diversity = usually better test accuracy")

## 📏 Step 5: Model Evaluation

### Confusion Matrix — OLD Way (Still Valid!)

---

### 🔑 Confusion Matrix Layout

```
                    PREDICTED
                  Fail    Pass
         Fail  [  TN  |  FP  ]
ACTUAL   Pass  [  FN  |  TP  ]
```

| Term | Stands For | Meaning |
|------|-----------|---------|
| **TP** | True Positive | Predicted Pass ✅, Actually Pass ✅ |
| **TN** | True Negative | Predicted Fail ✅, Actually Fail ✅ |
| **FP** | False Positive | Predicted Pass ❌, Actually Fail (Type 1 Error) |
| **FN** | False Negative | Predicted Fail ❌, Actually Pass (Type 2 Error) |

---

> 🎯 **What we learn here**: How to read a confusion matrix and understand model mistakes.

In [ ]:
# ═══════════════════════════════════════════════════
# 🔲  CONFUSION MATRIX — OLD WAY (Seaborn heatmap)
# ═══════════════════════════════════════════════════
# This approach was used before sklearn added ConfusionMatrixDisplay
# It still works fine and gives full customisation control.

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix (array):")
print(cm)
print(f"\n  TN (Predict Fail, Actually Fail) = {cm[0,0]}")
print(f"  FP (Predict Pass, Actually Fail) = {cm[0,1]}")
print(f"  FN (Predict Fail, Actually Pass) = {cm[1,0]}")
print(f"  TP (Predict Pass, Actually Pass) = {cm[1,1]}")

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Fail','Predicted Pass'],
            yticklabels=['Actual Fail','Actual Pass'],
            linewidths=1)
plt.title('Confusion Matrix — Old Way (Seaborn)', fontsize=12, fontweight='bold')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()
print("\n💡 Old way still works perfectly — full control over colours and labels!")

## 🆕 Confusion Matrix — NEW Way (ConfusionMatrixDisplay)

---

### 📖 Why Use the New Way?
- **Less code** — fewer lines needed
- **Built into sklearn** — no seaborn needed
- **`normalize='true'`** — shows percentages (great for imbalanced data)
- **`from_estimator()`** — directly from the trained model (even simpler!)

---

### Two Methods Available

| Method | Input | Use When |
|--------|-------|----------|
| `from_predictions(y_true, y_pred)` | Predictions array | You already have `y_pred` |
| `from_estimator(model, X, y)` | Model + data | Direct from model (simplest!) |

---

> 🎯 **What we learn here**: The modern sklearn way to display a confusion matrix — preferred in industry today.

In [ ]:
# ═══════════════════════════════════════════════════
# 🆕  CONFUSION MATRIX — NEW WAY (ConfusionMatrixDisplay)
# ═══════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ── Method 1: from_predictions (standard) ────────
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Fail','Pass'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('from_predictions()\n(standard)', fontweight='bold')

# ── Method 2: from_predictions (normalised) ──────
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Fail','Pass'],
    cmap='Greens', normalize='true', ax=axes[1]
)
axes[1].set_title('from_predictions(normalize=\'true\')\n(shows proportions)', fontweight='bold')

# ── Method 3: from_estimator (most direct) ───────
ConfusionMatrixDisplay.from_estimator(
    rf_clf, X_test, y_test,
    display_labels=['Fail','Pass'],
    cmap='Oranges', ax=axes[2]
)
axes[2].set_title('from_estimator()\n(directly from model — simplest!)', fontweight='bold')

plt.suptitle('Confusion Matrix — New Way (ConfusionMatrixDisplay)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("✅ ConfusionMatrixDisplay Advantages:")
print("   1. Simpler code")
print("   2. Built into sklearn (no extra imports)")
print("   3. normalize='true' shows percentages")
print("   4. from_estimator() is the most direct approach")

## 📋 Classification Report

---

### 🔑 Metrics Explained

**Precision** — Of all students we predicted would PASS, how many actually PASSED?
$$\text{Precision} = \frac{TP}{TP + FP}$$

**Recall** — Of all students who actually PASSED, how many did we correctly identify?
$$\text{Recall} = \frac{TP}{TP + FN}$$

**F1-Score** — Harmonic mean of Precision and Recall (balances both)
$$\text{F1} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

**Support** = Number of actual samples in that class

---

### 🎯 Which Metric to Focus On?

| Scenario | Focus Metric |
|----------|-------------|
| Medical (don't miss sick patients) | **Recall** (minimize FN) |
| Spam Filter (don't block good emails) | **Precision** (minimize FP) |
| Balanced problem | **F1-Score** or **Accuracy** |

In [ ]:
# ═══════════════════════════════════════════════════
# 📋  CLASSIFICATION REPORT
# ═══════════════════════════════════════════════════

print(classification_report(y_test, y_pred,
                             target_names=['Failed (0)','Passed (1)']))

print("💡 How to Read:")
print("  precision = Of predicted Passed, how many were right?")
print("  recall    = Of actual Passed, how many did we find?")
print("  f1-score  = Balance between precision and recall")
print("  support   = Actual count of each class in test set")
print("\n  'macro avg'    = Simple average of both classes")
print("  'weighted avg' = Weighted by support (class size)")

## 📈 ROC-AUC Score

---

### 🔑 New Words

| Term | Meaning |
|------|---------|
| **ROC** | Receiver Operating Characteristic — a performance curve |
| **AUC** | Area Under the Curve — a single number (0.5 → 1.0) |
| **TPR** | True Positive Rate = Recall = TP / (TP + FN) |
| **FPR** | False Positive Rate = FP / (FP + TN) |

---

### 📊 ROC Curve
- Plots **TPR (y-axis) vs FPR (x-axis)** at all possible thresholds
- **Perfect model** → curve goes to the top-left corner
- **Random model** → diagonal line, AUC = 0.5

### 🎯 Interpreting AUC

| AUC | Quality |
|-----|---------|
| 0.5 | Random — coin flip |
| 0.6–0.7 | Poor |
| 0.7–0.8 | Fair |
| 0.8–0.9 | Good ✅ |
| 0.9–1.0 | Excellent 🌟 |

---

> 💡 AUC is better than accuracy when classes are **imbalanced**!

In [ ]:
# ═══════════════════════════════════════════════════
# 📈  ROC-AUC CURVE
# ═══════════════════════════════════════════════════

# predict_proba → probability of each class (not just 0 or 1)
y_prob = rf_clf.predict_proba(X_test)[:, 1]  # probability of class 1 (Pass)

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_score   = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'darkorange', lw=2, label=f'ROC Curve (AUC = {roc_score:.3f})')
plt.plot([0,1],[0,1], 'navy', lw=2, linestyle='--', label='Random Classifier (AUC = 0.5)')
plt.fill_between(fpr, tpr, alpha=0.1, color='orange')
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR)', fontsize=12)
plt.title('ROC Curve — Random Forest', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

grade = ('Excellent 🌟' if roc_score>=0.9 else 'Good ✅' if roc_score>=0.8
         else 'Fair' if roc_score>=0.7 else 'Needs Improvement')
print(f"AUC = {roc_score:.3f} → {grade}")

## 🏆 Feature Importance

---

### 📖 What is Feature Importance?
> **Feature Importance** tells us which input columns had the most impact on the model's predictions.

---

### 🔄 How is it Calculated?
For each feature:
1. Find how much it **reduces Gini Impurity** when used to split nodes
2. Average across all trees
3. Normalise so all importances **sum to 1.0**

---

### 🎯 Why is it Useful?
1. **Understand your data** — what factors matter most?
2. **Feature Selection** — safely remove unimportant features
3. **Business Insight** — explain what drives the outcome to stakeholders

---

### 🔑 How to Access

```python
rf_model.feature_importances_   # returns an array, sums to 1.0
```

In [ ]:
# ═══════════════════════════════════════════════════
# 🏆  FEATURE IMPORTANCE
# ═══════════════════════════════════════════════════

fi = pd.DataFrame({
    'Feature'   : X.columns,
    'Importance': rf_clf.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance Scores:")
print(fi.to_string(index=False))
print(f"\nSum of all importances: {fi['Importance'].sum():.4f} (always = 1.0)")

# Plot
plt.figure(figsize=(9, 5))
colors = plt.cm.RdYlGn(fi['Importance'].values / fi['Importance'].max())
bars = plt.barh(fi['Feature'][::-1], fi['Importance'][::-1],
                color=colors[::-1], edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, fi['Importance'][::-1]):
    plt.text(bar.get_width()+0.003, bar.get_y()+bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=10)

plt.xlabel('Importance Score', fontsize=12)
plt.title('Feature Importance — Random Forest Classifier\n(Higher = More Important)',
          fontsize=13, fontweight='bold')
plt.xlim(0, fi['Importance'].max()+0.08)
plt.tight_layout(); plt.show()

print(f"\n🏆 Most Important : {fi.iloc[0]['Feature']}")
print(f"📉 Least Important: {fi.iloc[-1]['Feature']}")

## 🔧 Step 6: Hyperparameter Tuning — GridSearchCV

---

### 📖 What is GridSearchCV?
> **GridSearchCV** = Automatically tries **all combinations** of hyperparameters you specify, and finds the BEST one.

---

### 🔑 New Words

| Term | Meaning |
|------|---------|
| **Grid** | A set of parameter values to try |
| **CV** | Cross-Validation — split data into K folds to evaluate reliably |
| **K-Fold CV** | Split data into K parts, train K times, average the scores |

---

### 🔄 How GridSearchCV Works

```python
param_grid = {'n_estimators': [50, 100], 'max_depth': [3, None]}

# Combinations tried: (50,3), (50,None), (100,3), (100,None)
# Each with 5-fold CV = 4 × 5 = 20 model trainings

# Returns: Best combination + best score
```

---

### ⚡ Tip: `RandomizedSearchCV`
When you have many parameters, `GridSearchCV` can be slow.
Use `RandomizedSearchCV` — tries random combinations (much faster)!

In [ ]:
# ═══════════════════════════════════════════════════
# 🔧  GRIDSEARCHCV — HYPERPARAMETER TUNING
# ═══════════════════════════════════════════════════

param_grid = {
    'n_estimators'     : [50, 100, 150],
    'max_depth'        : [None, 5, 10],
    'max_features'     : ['sqrt', 'log2'],
    'min_samples_split': [2, 5]
}

total = 1
for v in param_grid.values():
    total *= len(v)
print(f"Total combinations: {total} × 5 folds = {total*5} model trainings")
print("Running GridSearchCV (may take ~30 seconds)...\n")

gs = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv      = 5,
    scoring = 'accuracy',
    n_jobs  = -1,     # use all CPU cores
    verbose = 0
)
gs.fit(X_train, y_train)

print("=" * 55)
print("✅ GridSearchCV Complete!")
print("=" * 55)
print("\n🏆 Best Parameters Found:")
for k, v in gs.best_params_.items():
    print(f"   {k:25s}: {v}")

best_rf = gs.best_estimator_
y_pred_best = best_rf.predict(X_test)
print(f"\n📊 Best CV Score : {gs.best_score_*100:.2f}%")
print(f"📊 Test Accuracy : {accuracy_score(y_test, y_pred_best)*100:.2f}%")
print(f"📊 Improvement   : {(accuracy_score(y_test, y_pred_best)-test_acc)*100:+.2f}% vs baseline")

---

# 🏠 Part 2: Random Forest for Regression

---

### 📖 What is Regression?
> **Regression** = Predicting a **continuous/numerical** value (not a category).

| Task | Model | Output |
|------|-------|--------|
| Pass or Fail? | Classifier | 0 or 1 |
| What price? | Regressor | ₹45,00,000 |

---

### 📊 Regression Evaluation Metrics

| Metric | Formula | Meaning |
|--------|---------|---------|
| **MAE** | mean(\|actual − predicted\|) | Average absolute error (easy to understand) |
| **MSE** | mean((actual − predicted)²) | Penalises large errors more |
| **RMSE** | √MSE | In same units as target (most common) |
| **R² Score** | 1 − SS_res/SS_tot | % of variance explained — 1.0 = perfect |

> 🎯 **R² = 0.85** means: "Our model explains 85% of the variation in house prices"

## 🏠 Creating Sample House Price Dataset

---

### Scenario: Predict House Prices in a City

| Column | Description |
|--------|-------------|
| `area_sqft` | Built area in square feet |
| `bedrooms` | Number of bedrooms |
| `bathrooms` | Number of bathrooms |
| `age_years` | Age of the house (older = cheaper) |
| `distance_km` | Distance from city centre |
| `parking` | Has covered parking? (0/1) |
| `price` *(target)* | House price in ₹ |

In [ ]:
# ═══════════════════════════════════════════════════
# 🏠  CREATE HOUSE PRICE DATASET
# ═══════════════════════════════════════════════════

np.random.seed(42)
n_h = 400

area        = np.random.uniform(500, 4000, n_h)
bedrooms    = np.random.randint(1, 6,    n_h)
bathrooms   = np.random.randint(1, 4,    n_h)
age         = np.random.uniform(0, 50,   n_h)
distance    = np.random.uniform(1, 30,   n_h)
parking     = np.random.randint(0, 2,    n_h)

df_h = pd.DataFrame({
    'area_sqft'  : area.round(0),
    'bedrooms'   : bedrooms,
    'bathrooms'  : bathrooms,
    'age_years'  : age.round(1),
    'distance_km': distance.round(1),
    'parking'    : parking
})

price = (2500*area + 150000*bedrooms + 80000*bathrooms
         - 10000*age - 15000*distance + 50000*parking
         + np.random.normal(0, 150000, n_h))
df_h['price'] = price.clip(300000, 15000000).round(-3)

print("🏠 House Price Dataset:")
display(df_h.head())
print(f"\nShape: {df_h.shape}")
print(f"Price range: ₹{df_h['price'].min():,.0f}  –  ₹{df_h['price'].max():,.0f}")
print(f"Mean price : ₹{df_h['price'].mean():,.0f}")

In [ ]:
# ─── EDA: Feature vs Price ───────────────────────────────────
feats_h = ['area_sqft','bedrooms','bathrooms','age_years','distance_km','parking']
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

for i, feat in enumerate(feats_h):
    ax = axes[i//3, i%3]
    ax.scatter(df_h[feat], df_h['price']/1e5, alpha=0.4,
               color='steelblue', s=20)
    # Trend line
    z = np.polyfit(df_h[feat], df_h['price'], 1)
    xs = np.linspace(df_h[feat].min(), df_h[feat].max(), 50)
    ax.plot(xs, np.poly1d(z)(xs)/1e5, 'r-', lw=2, alpha=0.7)
    ax.set_xlabel(feat); ax.set_ylabel('Price (₹ Lakhs)')
    ax.set_title(feat, fontweight='bold')

plt.suptitle('Feature vs House Price', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print("💡 area_sqft has STRONG positive effect on price")

In [ ]:
# ═══════════════════════════════════════════════════
# 🌳  RANDOM FOREST REGRESSOR
# ═══════════════════════════════════════════════════

X_h = df_h.drop('price', axis=1)
y_h = df_h['price']

X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42)

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_tr_h, y_tr_h)
y_pred_h = rf_reg.predict(X_te_h)

mae  = mean_absolute_error(y_te_h, y_pred_h)
mse  = mean_squared_error(y_te_h,  y_pred_h)
rmse = np.sqrt(mse)
r2   = r2_score(y_te_h, y_pred_h)

print("=" * 55)
print("🌳 RANDOM FOREST REGRESSOR — RESULTS")
print("=" * 55)
print(f"MAE  (Mean Absolute Error)  : ₹{mae:>12,.0f}")
print(f"MSE  (Mean Squared Error)   :  {mse:>18,.0f}")
print(f"RMSE (Root Mean Sq. Error)  : ₹{rmse:>12,.0f}")
print(f"R²   (R-Squared Score)      :  {r2:>14.4f}")
print("=" * 55)
print(f"\n💡 Our predictions are off by ₹{mae:,.0f} on average (MAE)")
print(f"   The model explains {r2*100:.1f}% of variation in house prices (R²)")

In [ ]:
# ─── Actual vs Predicted + Residuals ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Actual vs Predicted
lo = min(y_te_h.min(), y_pred_h.min()) / 1e5
hi = max(y_te_h.max(), y_pred_h.max()) / 1e5
axes[0].scatter(y_te_h/1e5, y_pred_h/1e5, alpha=0.4, color='steelblue', s=25)
axes[0].plot([lo,hi],[lo,hi],'r--',lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Price (₹ Lakhs)'); axes[0].set_ylabel('Predicted Price (₹ Lakhs)')
axes[0].set_title(f'Actual vs Predicted  (R²={r2:.3f})', fontweight='bold')
axes[0].legend()

# Residuals
residuals = (y_te_h - y_pred_h) / 1e5
axes[1].scatter(y_pred_h/1e5, residuals, alpha=0.4, color='coral', s=25)
axes[1].axhline(0, color='black', lw=2, ls='--')
axes[1].set_xlabel('Predicted (₹ Lakhs)'); axes[1].set_ylabel('Residual (₹ Lakhs)')
axes[1].set_title('Residual Plot\n(Should look random around 0)', fontweight='bold')

plt.suptitle('Regression Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("💡 Actual vs Predicted: Points close to diagonal = accurate predictions")
print("   Residual Plot: Random scatter = good model | Pattern = problem!")

In [ ]:
# ─── Feature Importance for Regression ───────────────────────
fi_h = pd.DataFrame({
    'Feature'   : X_h.columns,
    'Importance': rf_reg.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(fi_h['Feature'], fi_h['Importance'],
         color='steelblue', edgecolor='black', linewidth=0.5)
for i, val in enumerate(fi_h['Importance']):
    plt.text(val+0.002, i, f'{val:.3f}', va='center', fontsize=10)
plt.xlabel('Importance Score');
plt.title('Feature Importance — House Price Regressor', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"\n🏆 Most Important: {fi_h.iloc[-1]['Feature']}")
print("   → This feature drives house price predictions the most!")

---

## 🎓 Summary — What We Covered!

---

### Random Forest Core Concepts

| Concept | Key Points |
|---------|------------|
| **Random Forest** | Ensemble of many Decision Trees |
| **Bootstrapping** | Sampling with replacement — creates diverse datasets |
| **OOB Error** | Free validation from ~37% unseen samples per tree |
| **max_features** | Random feature subset at each split — creates diversity! |
| **n_estimators** | More trees = better (diminishing returns after ~200) |

---

### Hyperparameters to Remember

```python
RandomForestClassifier(
    n_estimators       = 100,     # number of trees
    max_depth          = None,    # None = fully grown (can overfit)
    max_features       = 'sqrt',  # features per split
    min_samples_split  = 2,       # min samples to split a node
    oob_score          = True,    # free OOB validation
    n_jobs             = -1,      # use all CPU cores
    random_state       = 42       # reproducibility
)
```

---

### Evaluation Metrics

| Task | Metrics |
|------|---------|
| **Classification** | Accuracy, Confusion Matrix, Precision, Recall, F1, ROC-AUC |
| **Regression** | MAE, MSE, RMSE, R² |

---

### Modern sklearn Tools (Use These in Industry!)
- `ConfusionMatrixDisplay.from_predictions()` — simple confusion matrix
- `ConfusionMatrixDisplay.from_estimator()` — directly from model
- `GridSearchCV` — automated hyperparameter tuning

---

> 🚀 **Next**: Open **Notebook 2** to apply everything on **real Kaggle datasets**!